# Compare two CLI geometries

Fresh exact-excluded detailed stem4 and retained advanced vcdr12 outputs. The explicit benchmark option removes exact full-junction reference keys; no score or calibration is transferred between geometries.

The displayed scores and their arithmetic delta are geometry diagnostics only. Distinct geometries do not have calibrated comparable score scales; the delta is neither accuracy nor a confidence improvement.

In [ ]:
# Identify the installed CLI and print the fresh kernel environment.
import os, sys, subprocess, tempfile, json
from pathlib import Path
from importlib.metadata import version
os.environ["POLARS_MAX_THREADS"] = "1"
print("Python:", sys.version, "CLI interpreter:", sys.executable)
for package in ("vdjmatch", "vdjtools", "seqtree", "polars"):
    print(package, version(package))
# Open this notebook from the repository root or the examples directory.
examples = next((p for p in (Path.cwd()/"examples", Path.cwd())
                 if (p/"inputs/reference.tsv").is_file()), None)
if examples is None:
    raise RuntimeError("Use the downloaded complete examples bundle, not this notebook alone")
# Deterministic demonstration: no sampling or RNG seeds are used.
output = Path(tempfile.mkdtemp(prefix="vdjmatch-example-"))/"fresh"
output.mkdir()
print("Fresh output directory:", output)


In [ ]:
# Calculate each geometry freshly through this kernel's installed CLI.
subprocess.run([sys.executable, "-m", "vdjmatch.cli", "annotate",
                str(examples/"inputs/cohort/sample_a.tsv"),
                "--vdjdb", str(examples/"inputs/reference.tsv"), "--threads", "4",
                "--exclude-exact", "--details", "--output-prefix", str(output/"stem4")],
               check=True, timeout=120)
subprocess.run([sys.executable, "-m", "vdjmatch.cli", "annotate",
                str(examples/"inputs/cohort/sample_a.tsv"),
                "--vdjdb", str(examples/"inputs/reference.tsv"), "--threads", "4",
                "--exclude-exact", "--details", "--v-regions", "cdr12", "--stem-depth", "0",
                "--distance-matrix", "tcrdist", "--junction-ends", "trim3",
                "--output-prefix", str(output/"vcdr12")], check=True, timeout=120)

In [ ]:
# Join detailed CLI evidence on source identity and the complete pMHC key.
import polars as pl
keys = ["sample_id", "source_sequence_id", "epitope", "mhc_a", "mhc_b", "mhc_class"]
tables = {name: pl.read_csv(output/(name+".enrichment.tsv"), separator="\t")
          for name in ("stem4", "vcdr12")}
assert all(table.select(keys).unique().height == table.height for table in tables.values())
left = tables["stem4"].select(keys+["kernel_score"]).rename({"kernel_score":"stem4_score"})
right = tables["vcdr12"].select(keys+["kernel_score"]).rename({"kernel_score":"vcdr12_score"})
comparison = left.join(right, on=keys, how="full", coalesce=True, validate="1:1", nulls_equal=True)
comparison = comparison.with_columns((pl.col("vcdr12_score")-pl.col("stem4_score")).alias("score_delta"))
print(comparison.sort(keys))
assert comparison.height == left.height == right.height
for name in tables:
    print(name, json.loads((output/(name+".manifest.json")).read_text())["kernel"])


All displayed results come from fresh CLI tables. Raw synthetic `DEMO_A`/`DEMO_B` labels are software examples, not experimental binding truth. Keep no-hits and missing-chain dispositions. This example establishes neither accuracy nor a false-call threshold. Inspect the CLI manifests beside the tables for model, reference and resource settings. No computed outputs are committed or reused.